### Imports

In [ ]:
import numpy as np
import open3d as o3d
from plicalib import io as plica_io
from plicalib import clean as plica_clean
from plicalib import meshes
import plotly.graph_objects as go
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path

def _cut_and_stitch(vertices, triangles, n, p, inverse_cut=False):
    vertices = np.asarray(vertices)
    triangles = np.asarray(triangles)

    norm = np.linalg.norm(n)
    if norm < 1e-10:
        return vertices.copy(), triangles.copy()

    n_unit = n / norm
    signed_dist = (vertices - p) @ n_unit

    if inverse_cut:
        signed_dist = -signed_dist

    above = signed_dist >= 0

    i0 = triangles[:, 0]
    i1 = triangles[:, 1]
    i2 = triangles[:, 2]

    fully_kept = above[i0] & above[i1] & above[i2]
    fully_cut = ~above[i0] & ~above[i1] & ~above[i2]
    partial = ~fully_kept & ~fully_cut

    def _compact_mesh(kept_tris):
        if len(kept_tris) == 0:
            return (
                np.empty((0, 3), dtype=vertices.dtype),
                np.empty((0, 3), dtype=triangles.dtype),
            )

        used = np.unique(kept_tris)
        remap = {int(old): new for new, old in enumerate(used)}

        new_tris = np.array(
            [[remap[int(a)], remap[int(b)], remap[int(c)]] for a, b, c in kept_tris],
            dtype=triangles.dtype,
        )

        return vertices[used].copy(), new_tris

    if not partial.any():
        return _compact_mesh(triangles[fully_kept])

    partial_triangles = triangles[partial]

    segments, plane_ids, tri_ids, crossed_edges = meshes.plane_slice_single_normal(
        vertices,
        partial_triangles,
        plane_origins=p[None, :],
        plane_normal=n_unit,
        return_crossed_edges=True,
    )

    segments = np.asarray(segments).reshape(-1, 2, 3)
    tri_ids = np.asarray(tri_ids, dtype=int).reshape(-1)
    crossed_edges = np.asarray(crossed_edges).reshape(-1, 2, 2)

    if len(segments) == 0:
        return _compact_mesh(triangles[fully_kept])

    if not (len(segments) == len(tri_ids) == len(crossed_edges)):
        raise ValueError(
            "Inconsistent slicing output sizes: "
            f"segments={len(segments)}, "
            f"tri_ids={len(tri_ids)}, "
            f"crossed_edges={len(crossed_edges)}"
        )

    def _edge_key(a, b):
        a = int(a)
        b = int(b)
        return (a, b) if a <= b else (b, a)

    new_verts = []
    new_tris = []
    remap = {}

    def _get_or_add(orig_idx):
        orig_idx = int(orig_idx)

        if orig_idx not in remap:
            remap[orig_idx] = len(new_verts)
            new_verts.append(vertices[orig_idx])

        return remap[orig_idx]

    # Add fully kept triangles first.
    for tri in triangles[fully_kept]:
        new_tris.append(
            [
                _get_or_add(tri[0]),
                _get_or_add(tri[1]),
                _get_or_add(tri[2]),
            ]
        )

    # Add clipped partial triangles.
    for k, tri_idx in enumerate(tri_ids):
        tri = partial_triangles[tri_idx]
        s = signed_dist[tri]
        above_mask = s >= 0
        n_above = int(above_mask.sum())

        if n_above == 0 or n_above == 3:
            continue

        p0_idx = len(new_verts)
        new_verts.append(segments[k, 0])

        p1_idx = len(new_verts)
        new_verts.append(segments[k, 1])

        # crossed_edges[k] has shape (2, 2), already global vertex ids.
        cut_point_of_edge = {}

        e0 = crossed_edges[k, 0]
        e1 = crossed_edges[k, 1]

        cut_point_of_edge[_edge_key(e0[0], e0[1])] = p0_idx
        cut_point_of_edge[_edge_key(e1[0], e1[1])] = p1_idx

        if n_above == 1:
            # One kept vertex gives one triangle.
            ki = int(np.where(above_mask)[0][0])

            vk = _get_or_add(tri[ki])

            e_forward = _edge_key(tri[ki], tri[(ki + 1) % 3])
            e_backward = _edge_key(tri[(ki - 1) % 3], tri[ki])

            pa = cut_point_of_edge[e_forward]
            pb = cut_point_of_edge[e_backward]

            new_tris.append([vk, pa, pb])

        elif n_above == 2:
            # Two kept vertices give a clipped quad, triangulated into two triangles.
            li = int(np.where(~above_mask)[0][0])

            e_backward = _edge_key(tri[(li - 1) % 3], tri[li])
            e_forward = _edge_key(tri[li], tri[(li + 1) % 3])

            pa = cut_point_of_edge[e_backward]
            pb = cut_point_of_edge[e_forward]

            kv_next = _get_or_add(tri[(li + 1) % 3])
            kv_prev = _get_or_add(tri[(li - 1) % 3])

            new_tris.append([kv_next, kv_prev, pa])
            new_tris.append([kv_next, pa, pb])

    new_vertices = np.asarray(new_verts, dtype=vertices.dtype)
    new_triangles = np.asarray(new_tris, dtype=triangles.dtype)

    return new_vertices, new_triangles
def _run_cut_mesh(mesh_db, default_n=(0.0, 0.0, 1.0)):
    if len(mesh_db) == 0:
        raise ValueError("mesh_db is empty")

    cut_meshes = {}
    state = {"suspend": False}

    fig = go.FigureWidget()

    fig.update_layout(
        width=800,
        height=700,
        scene=dict(
            xaxis_title="x",
            yaxis_title="y",
            zaxis_title="z",
            aspectmode="data",
            uirevision="keep",
        ),
        margin=dict(l=0, r=0, t=0, b=0),
        legend=dict(itemsizing="constant", y=0.5),
    )

    palette = px.colors.qualitative.T10

    default_n = np.asarray(default_n, dtype=float)
    if np.linalg.norm(default_n) < 1e-12:
        default_n = np.array([0.0, 0.0, 1.0])

    def _make_slider(desc, val, mn, mx, step):
        return widgets.FloatSlider(
            value=val,
            min=mn,
            max=mx,
            step=step,
            description=desc,
            continuous_update=False,
            style={"description_width": "40px"},
            layout=widgets.Layout(width="300px"),
        )

    sample_id_widget = widgets.Dropdown(
        options=list(mesh_db.keys()),
        description="Sample",
    )

    nx_widget = _make_slider("n_x", float(default_n[0]), -1.0, 1.0, 0.01)
    ny_widget = _make_slider("n_y", float(default_n[1]), -1.0, 1.0, 0.01)
    nz_widget = _make_slider("n_z", float(default_n[2]), -1.0, 1.0, 0.01)

    px_widget = _make_slider("p_x", 0.0, -1.0, 1.0, 1.0)
    py_widget = _make_slider("p_y", 0.0, -1.0, 1.0, 1.0)
    pz_widget = _make_slider("p_z", 0.0, -1.0, 1.0, 1.0)

    inverse_cut_widget = widgets.Checkbox(value=False, description="Inverse cut")
    cut_button_widget = widgets.Button(description="Cut", button_style="primary")
    close_button_widget = widgets.Button(description="Close")

    row1_widget = widgets.HBox(
        [sample_id_widget, close_button_widget],
        layout=widgets.Layout(
            width="100%",
            justify_content="space-between",
            align_items="center",
        ),
    )

    row2_left_widget = widgets.VBox(
        [
            widgets.Label("Normal vector (n)"),
            nx_widget,
            ny_widget,
            nz_widget,
            widgets.Label("Point on plane (p)"),
            px_widget,
            py_widget,
            pz_widget,
            inverse_cut_widget,
            cut_button_widget,
        ],
        layout=widgets.Layout(
            width="100%",
            height="680px",
            overflow_y="auto",
            overflow_x="hidden",
            align_self="flex-start",
        ),
    )

    row2_right_widget = widgets.Box(
        [fig],
        layout=widgets.Layout(
            width="100%",
            height="680px",
            align_self="stretch",
            overflow="visible",
        ),
    )

    grid = widgets.GridspecLayout(2, 2, width="100%", grid_gap="12px")
    grid[0, :] = row1_widget
    grid[1, 0] = row2_left_widget
    grid[1, 1] = row2_right_widget
    grid.layout.grid_template_columns = "320px 1fr"
    grid.layout.grid_template_rows = "auto 1fr"
    grid.layout.align_items = "flex-start"

    # -------------------------------------------------------------------------
    # Helpers
    # -------------------------------------------------------------------------

    def _get_n():
        n = np.array(
            [nx_widget.value, ny_widget.value, nz_widget.value],
            dtype=float,
        )

        norm = np.linalg.norm(n)

        if norm < 1e-12:
            n = default_n.copy()
            norm = np.linalg.norm(n)

        if norm < 1e-12:
            n = np.array([0.0, 0.0, 1.0])
            norm = 1.0

        return n / norm

    def _get_p():
        return np.array(
            [px_widget.value, py_widget.value, pz_widget.value],
            dtype=float,
        )

    def _set_slider_range(widget, mn, mx, value, n_steps=200):
        mn = float(mn)
        mx = float(mx)
        value = float(value)

        if not np.isfinite(mn) or not np.isfinite(mx):
            raise ValueError("Slider range contains non-finite values")

        if mx < mn:
            mn, mx = mx, mn

        if abs(mx - mn) < 1e-12:
            pad = max(1.0, abs(mn) * 0.05)
            mn -= pad
            mx += pad

        value = float(np.clip(value, mn, mx))
        step = (mx - mn) / n_steps

        # Expand first, otherwise ipywidgets can complain when shrinking range.
        old_min = float(widget.min)
        old_max = float(widget.max)
        old_val = float(widget.value)

        widget.min = min(old_min, mn, old_val, value)
        widget.max = max(old_max, mx, old_val, value)

        widget.step = step
        widget.value = value

        widget.min = mn
        widget.max = mx

    def _update_p_slider_ranges():
        name = sample_id_widget.value
        verts = mesh_db[name]["vertices"]

        for widget, axis in [
            (px_widget, 0),
            (py_widget, 1),
            (pz_widget, 2),
        ]:
            mn = float(verts[:, axis].min())
            mx = float(verts[:, axis].max())
            mid = 0.5 * (mn + mx)

            _set_slider_range(widget, mn, mx, mid)

    def _reset_widgets_to_defaults():
        state["suspend"] = True

        nx_widget.value = float(default_n[0])
        ny_widget.value = float(default_n[1])
        nz_widget.value = float(default_n[2])
        inverse_cut_widget.value = False

        _update_p_slider_ranges()

        state["suspend"] = False

    def _plane_basis(n):
        refs = np.eye(3)
        ref = refs[np.argmin(np.abs(refs @ n))]

        u = np.cross(n, ref)
        u_norm = np.linalg.norm(u)

        if u_norm < 1e-12:
            ref = np.array([1.0, 0.0, 0.0])
            u = np.cross(n, ref)
            u_norm = np.linalg.norm(u)

        u = u / u_norm
        v = np.cross(n, u)
        v = v / np.linalg.norm(v)

        return u, v

    # -------------------------------------------------------------------------
    # Plotting
    # -------------------------------------------------------------------------

    def _plot_mesh(actually_cut):
        name = sample_id_widget.value
        vertices = mesh_db[name]["vertices"]
        triangles = mesh_db[name]["triangles"]

        if actually_cut:
            n = _get_n()
            p = _get_p()

            cut_verts, cut_tris = _cut_and_stitch(
                vertices,
                triangles,
                n,
                p,
                inverse_cut=inverse_cut_widget.value,
            )

            v = cut_verts
            t = cut_tris

            cut_meshes[name] = {
                "vertices": cut_verts,
                "triangles": cut_tris,
                "n": n.copy(),
                "p": p.copy(),
                "inverse_cut": bool(inverse_cut_widget.value),
            }

        else:
            v = vertices
            t = triangles

        fig.add_trace(
            go.Mesh3d(
                x=v[:, 0],
                y=v[:, 1],
                z=v[:, 2],
                i=t[:, 0],
                j=t[:, 1],
                k=t[:, 2],
                color=palette[0],
                opacity=0.9,
                name="mesh",
                showscale=False,
            )
        )

    def _update_overlay():
        if len(fig.data) == 0:
            return

        p = _get_p()
        n = _get_n()

        name = sample_id_widget.value
        verts = mesh_db[name]["vertices"]

        extent = float(np.linalg.norm(verts.max(axis=0) - verts.min(axis=0)))
        if extent < 1e-12:
            extent = 1.0

        cone_len = extent * 0.08
        plane_size = extent * 0.4

        u, v = _plane_basis(n)

        corners = np.array(
            [
                p + plane_size * (-u - v),
                p + plane_size * (+u - v),
                p + plane_size * (-u + v),
                p + plane_size * (+u + v),
            ]
        )

        xs = corners[:, 0].reshape(2, 2)
        ys = corners[:, 1].reshape(2, 2)
        zs = corners[:, 2].reshape(2, 2)

        with fig.batch_update():
            for trace in fig.data:
                if trace.name == "plane_origin":
                    trace.update(
                        x=[p[0]],
                        y=[p[1]],
                        z=[p[2]],
                        mode="markers",
                        marker=dict(size=6, color="red"),
                    )

                elif trace.name == "plane_normal":
                    trace.update(
                        x=[p[0]],
                        y=[p[1]],
                        z=[p[2]],
                        u=[n[0] * cone_len],
                        v=[n[1] * cone_len],
                        w=[n[2] * cone_len],
                        sizemode="absolute",
                        sizeref=cone_len,
                        colorscale=[[0, "red"], [1, "red"]],
                        showscale=False,
                    )

                elif trace.name == "plane_surface":
                    trace.update(
                        x=xs,
                        y=ys,
                        z=zs,
                        colorscale=[
                            [0, "rgb(255,50,50)"],
                            [1, "rgb(255,50,50)"],
                        ],
                        opacity=0.25,
                        surfacecolor=np.zeros((2, 2)),
                        showscale=False,
                    )

    def _full_replot(actually_cut):
        with fig.batch_update():
            fig.data = ()

            _plot_mesh(actually_cut)

            fig.add_trace(
                go.Scatter3d(
                    x=[],
                    y=[],
                    z=[],
                    name="plane_origin",
                    mode="markers",
                    marker=dict(size=6, color="red"),
                    showlegend=False,
                )
            )

            fig.add_trace(
                go.Cone(
                    x=[],
                    y=[],
                    z=[],
                    u=[],
                    v=[],
                    w=[],
                    name="plane_normal",
                    showscale=False,
                    colorscale=[[0, "red"], [1, "red"]],
                    showlegend=False,
                )
            )

            fig.add_trace(
                go.Surface(
                    x=np.zeros((2, 2)),
                    y=np.zeros((2, 2)),
                    z=np.zeros((2, 2)),
                    name="plane_surface",
                    showscale=False,
                    opacity=0.25,
                    surfacecolor=np.zeros((2, 2)),
                    colorscale=[
                        [0, "rgb(255,50,50)"],
                        [1, "rgb(255,50,50)"],
                    ],
                    showlegend=False,
                )
            )

        _update_overlay()

    # -------------------------------------------------------------------------
    # Observers
    # -------------------------------------------------------------------------

    def _on_sample_change(change):
        if change["name"] != "value":
            return

        if change["new"] is None:
            return

        _reset_widgets_to_defaults()
        _full_replot(False)

    def _on_np_slider_change(change):
        if state["suspend"]:
            return

        if change["name"] != "value":
            return

        _update_overlay()

    def _on_cut(b):
        _full_replot(True)

    def _on_close(b):
        try:
            fig.close()
        except Exception:
            pass

        grid.close()

    cut_button_widget.on_click(_on_cut)
    close_button_widget.on_click(_on_close)

    sample_id_widget.observe(_on_sample_change, names="value")

    for w in [
        nx_widget,
        ny_widget,
        nz_widget,
        px_widget,
        py_widget,
        pz_widget,
    ]:
        w.observe(_on_np_slider_change, names="value")

    # -------------------------------------------------------------------------
    # Initial draw
    # -------------------------------------------------------------------------

    sample_id_widget.value = list(mesh_db.keys())[0]

    _reset_widgets_to_defaults()
    _full_replot(False)

    display(grid)

    return cut_meshes

### Slice meshes

In [ ]:
root_path = Path("examples/legdisc/")
metadata_file = root_path.joinpath("legdisc.json")
root_path = Path("examples/wingdisc/wildtype/")
metadata_file = root_path.joinpath("wildtype.json")
mesh_db = plica_io.from_json_database(metadata_file, separate_into_vertices_and_faces=True, root_path=root_path)

In [ ]:
cut_meshes = _run_cut_mesh(mesh_db)

In [ ]:
save_path = root_path.joinpath("cut/")
save_path.mkdir(exist_ok=True)
save_outlines = True
for name, cut_mesh in cut_meshes.items():
    verts = cut_mesh["vertices"]
    tris = cut_mesh["triangles"]
    p = cut_mesh["p"]
    n = cut_mesh["n"]
    paths, tri_paths = meshes.plane_slice_paths(mesh_db[name]["vertices"], mesh_db[name]["triangles"], p, n)
    savez_kwargs = {}
    if save_outlines:
        for i in range(len(paths)):
            savez_kwargs[f"path_{i}"] = paths[i]
            savez_kwargs[f"tri_path_{i}"] = tri_paths[i]
    np.savez(
        save_path.joinpath(f"{name}.npz"),
        vertices=verts,
        triangles=tris,
        plane_point=p,
        plane_normal=n,
        **savez_kwargs
    )